# Exercise 1 — Machine learning, regression, trees & forests

This exercise accompanies **Lecture 1 (Oct 19), Sessions 2–4**. You will:

- implement **linear regression from scratch** — MSE, the normal equations and gradient descent — and compare it with scikit-learn,
- implement **logistic regression from scratch** — sigmoid, cross-entropy and gradient descent — and compare it with scikit-learn,
- apply **decision trees and random forests** with scikit-learn (datasets are set up for you),
- evaluate models honestly with train/test splits, R², RMSE and confusion matrices.

**Datasets:** synthetic regression/blobs/moons plus `load_diabetes`, `load_wine`
and `load_breast_cancer` — all bundled with scikit-learn, no downloads needed.

## Setup

Run the cell below, then work through the parts in order. Add your own code
cells below each starter — there are no hidden tests; your printed output and
plots are the feedback.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import (
    make_regression,
    load_diabetes,
    make_blobs,
    make_moons,
    load_wine,
    load_breast_cancer,
)
from sklearn.linear_model import LinearRegression, SGDRegressor, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

## Part A — Linear regression by hand (vs. scikit-learn)

### A.1 — A synthetic straight-line dataset

In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 60)
y = 1.5 + 2.0 * x + rng.normal(scale=2.0, size=x.size)
X = x.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color="#2563eb", s=24)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("A noisy straight-line relationship")
plt.show()

### A.2 — The mean squared error

$$\mathcal{L}(w, b) = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)^2,
\qquad \hat{y} = Xw + b$$

In [ ]:
# YOUR CODE HERE
def mse(X, y, w, b):
    """Mean squared error of the linear model y_hat = X @ w + b."""
    ...

### A.3 — Closed form: the normal equations

The MSE is convex, so setting its gradient to zero gives the global minimum.
Augment $X$ with a column of ones and solve

$$w = (X^\top X)^{-1} X^\top y$$

(`np.linalg.lstsq` does this stably via the pseudo-inverse.)

In [ ]:
# YOUR CODE HERE
def fit_closed_form(X, y):
    """Return (w, b) that minimise the MSE using the normal equations.

    Hint: build an augmented matrix A = [X, ones], solve A^T A theta = A^T y,
    then split theta into w and b.
    """
    ...

### A.4 — Gradient descent

$$\nabla_w \mathcal{L} = \frac{2}{n} X^\top (\hat{y} - y),
\qquad \nabla_b \mathcal{L} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

Update `w ← w − lr · grad_w`, `b ← b − lr · grad_b`, repeated for `n_iter` epochs.

In [ ]:
# YOUR CODE HERE
def fit_linear_gd(X, y, lr=0.05, n_iter=1000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = []
    for _ in range(n_iter):
        # compute predictions, the two gradients, update w and b,
        # and append the current MSE to `losses`
        ...
    return w, b, losses


w_gd, b_gd, losses = fit_linear_gd(X, y)
print(f"w = {w_gd}, b = {b_gd:+.3f}")

### A.5 — Compare with scikit-learn

Fit `LinearRegression` (closed form) and `SGDRegressor` (gradient descent) on
the same data. Split 70/30 with `random_state=0` and report **R²** and **RMSE**
for all three models.

In [ ]:
# YOUR CODE HERE: fit both sklearn models, print w and b for all three,
# then print test R² and RMSE (use r2_score and mean_squared_error)
...

### A.6 — (Bonus) A real regression dataset

`load_diabetes()` ships 442 patients and 10 standardised features. Fit a
`LinearRegression`, report test R² and RMSE, and plot true-vs-predicted.

In [ ]:
# YOUR CODE HERE
...

## Part B — Logistic regression by hand (vs. scikit-learn)

### B.1 — A synthetic two-class dataset

In [ ]:
Xb, yb = make_blobs(
    n_samples=200, centers=[(-2.0, -2.0), (2.0, 2.0)],
    cluster_std=1.2, random_state=0,
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(Xb[yb == 0, 0], Xb[yb == 0, 1], color="#dc2626", alpha=0.7, label="class 0")
ax.scatter(Xb[yb == 1, 0], Xb[yb == 1, 1], color="#16a34a", alpha=0.7, label="class 1")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
ax.set_aspect("equal")
plt.show()

### B.2 — The sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

*Check:* `sigmoid(0)` should be `0.5`.

In [ ]:
# YOUR CODE HERE
def sigmoid(z):
    ...

### B.3 — The cross-entropy loss

$$\mathcal{L}(w, b) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right]$$

In [ ]:
# YOUR CODE HERE
def cross_entropy_loss(X, y, w, b):
    ...

### B.4 — Train with gradient descent

$$\nabla_w \mathcal{L} = \frac{1}{n} X^\top (\hat{p} - y), \qquad
\nabla_b \mathcal{L} = \frac{1}{n} \sum_i (\hat{p}_i - y_i)$$

In [ ]:
def fit_logistic_gd(X, y, lr=0.5, n_iter=500):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = []
    for _ in range(n_iter):
        # YOUR CODE HERE: compute p, the two gradients, update w and b,
        # and append the current cross-entropy loss to `losses`
        ...
    return w, b, losses


w_hand, b_hand, losses_hand = fit_logistic_gd(Xb, yb)
print(f"learned w = {w_hand}, b = {b_hand:+.3f}")

### B.5 — Compare against scikit-learn

Fit `LogisticRegression` on the same data and plot the two decision boundaries.
> **Note:** scikit-learn applies L2 regularisation by default, so its weights
> are slightly smaller — that is expected.

In [ ]:
# YOUR CODE HERE: fit sklearn, print coef_/intercept_/accuracy, then draw both
# boundaries with a contour plot (see the lecture for the meshgrid recipe)
...

## Part C — Decision trees & random forests (scikit-learn)

The **wine** dataset: 178 samples, 13 chemical features, 3 grape varieties.

### C.1 — Load and split (provided)

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(
    X_w, y_w, test_size=0.3, random_state=0, stratify=y_w
)
print(f"train: {Xw_tr.shape}, test: {Xw_te.shape}, classes: {list(wine.target_names)}")

### C.2 — An unlimited tree overfits

Fit `DecisionTreeClassifier()` with **no** depth limit. Compare train and test
accuracy — how big is the gap?

In [ ]:
# YOUR CODE HERE
...

### C.3 — Tuning `max_depth`

Loop over depths 1–20, record train and test accuracy, and plot both curves.
Where does test accuracy peak?

In [ ]:
# YOUR CODE HERE
...

### C.4 — Single tree vs random forest

Compare, with 5-fold cross-validation (`cross_val_score`):

1. `DecisionTreeClassifier(max_depth=3)`
2. `RandomForestClassifier(n_estimators=200, random_state=0)`

Report mean ± std of the CV accuracy for both.

In [ ]:
# YOUR CODE HERE
# Expected: the forest wins by a comfortable margin
...

### C.5 — What did the forest learn?

Fit a forest on the training data and plot `feature_importances_` as a
horizontal bar chart (label the bars with `wine.feature_names`). Which three
features matter most?

In [ ]:
# YOUR CODE HERE
...

### C.6 — Where is it wrong?

Show the confusion matrix of the forest on the test set. Which classes get
confused with each other?

In [ ]:
# YOUR CODE HERE: ConfusionMatrixDisplay.from_estimator or from_predictions
...

### C.7 — (Bonus) breast cancer: trees vs the linear model

Rerun the linear model from Part B on `load_breast_cancer()` and compare it
with a `RandomForestClassifier` (5-fold CV). Which family wins here?

In [ ]:
# YOUR CODE HERE
...

---

Well done — you implemented **linear** and **logistic regression by hand**, and
benchmarked **trees and forests** with scikit-learn. Next: **neural networks**
(`notebooks/02/neural_networks_exercise.ipynb`).